In [2]:
!pip install langchain-google-genai langchain-community chromadb pandas openpyxl
!pip uninstall langchain -y
!pip install langchain
!pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 

Found existing installation: langchain 1.3.18
Uninstalling langchain-1.3.18:
  Successfully uninstalled langchain-1.3.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 22.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
import os

# LangChain imports
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [4]:
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass(
    "Enter your Gemini API Key: "
)

print("API key configured successfully.")


Enter your Gemini API Key: ··········
API key configured successfully.


In [5]:
# File names
policy_file = "policy.txt"
orders_file = "orders.xlsx"
products_file = "products.xlsx"

# Check that files exist
import os

for file in [policy_file, orders_file, products_file]:
    if os.path.exists(file):
        print(f"✓ {file} found")
    else:
        print(f"✗ {file} NOT found")

✗ policy.txt NOT found
✗ orders.xlsx NOT found
✗ products.xlsx NOT found


In [10]:
# Load policy
with open("policy.txt", "r", encoding="utf-8") as f:
    policy_text = f.read()


import pandas as pd
# Load Excel files
products_df = pd.read_excel("products.xlsx")

print("Policy loaded:", len(policy_text), "characters")

print("Products:", products_df.shape)

Policy loaded: 2665 characters
Products: (23, 7)


In [12]:
from langchain_core.documents import Document

# =====================================
# POLICY DOCUMENTS ONLY IN CHROMA
# (Products & Google Sheets data are NEVER indexed in Chroma;
# they are fetched live in real time to reflect price/stock changes immediately!)
# =====================================

documents = [
    Document(
        page_content=policy_text,
        metadata={
            "source": "policy.txt",
            "type": "policy"
        }
    )
]

print("Total policy documents indexed in Chroma:", len(documents))


Total policy documents indexed in Chroma: 1


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Total chunks: 27

--- Chunk 1 ---
COMPANY POLICY DOCUMENT
TechMart Online Store — Customer Policies

PRODUCT CATEGORIES
TechMart offers a wide range of electronics and accessories. Our current product categories include:
- Laptops: Premium and gaming laptops from Apple, Dell, Lenovo, HP, and more.
- Mobile Phones: Flagship smartphones from Samsung, Apple, Google, OnePlus, and more.
- Tablets: Tablets from Apple, Samsung, Xiaomi, and more.
- Headphones: Over-ear and wireless headphones from Sony, Apple, Bose, and more.
- Speakers: Portable and indoor speakers from JBL, Sony, Bose, and more.
- Keyboards: Mechanical and wireless keyboards from Logitech, Keychron, Razer, and more.
- Mouse: Ergonomic and gaming mice from Logitech, Razer, SteelSeries, and more.

--- Chunk 2 ---
RETURN POLICY
Customers may return items within 30 days of the purchase date.
All returned items must be unused and in their original packaging with all accessories included.
Electronics such as laptops, phones, and t

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embeddings model initialized.")

Embeddings model initialized.


In [15]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="techmart_rag"
)

print("Chroma vector database created successfully!")

Chroma vector database created successfully!


In [16]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Retriever ready!")

Retriever ready!


In [17]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

print("Gemini 3.6 Flash is ready!")

Gemini 3.6 Flash is ready!


In [18]:
def get_live_products():
    """Fetch live product inventory in real time (Zero Chroma caching)"""
    try:
        # Live Google Sheet cloud link (reflects real-time price & stock updates immediately)
        url = "https://docs.google.com/spreadsheets/d/1UfEYYRrUWIWbD3VOBXIUuWraFRkbe3uh/export?format=csv&gid=1097220337"
        return pd.read_csv(url)
    except Exception:
        return pd.read_excel("products.xlsx")

def rag_chat(question):
    # 1. Fetch live product inventory in REAL TIME (never stale, zero Chroma indexing)
    live_df = get_live_products()
    live_inventory_context = "\n".join(
        f"- {row['Model Name']} ({row['Brand']}): Price ${row['price']}, Stock: {row['stock']} units, Category: {row['Category']}"
        for _, row in live_df.iterrows()
    )

    # 2. Retrieve ONLY company policy passages from Chroma vector database
    retrieved_docs = retriever.invoke(question)
    policy_context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # 3. Grounded prompt with live inventory and retrieved policy
    prompt = f"""
You are TechMart AI, a professional customer support assistant.

LIVE REAL-TIME INVENTORY (Always up-to-date from Google Sheets, zero vector caching):
{live_inventory_context}

RETRIEVED COMPANY POLICIES (From Chroma Vector Store):
{policy_context}

Customer Question:
{question}

Professional Answer:
"""

    # Get Gemini response
    response = llm.invoke(prompt)

    content = response.content
    if isinstance(content, str):
        answer = content
    elif isinstance(content, list):
        text_parts = [item.get("text", "") for item in content if isinstance(item, dict) and item.get("text")]
        answer = "\n".join(text_parts)
    else:
        answer = str(content)

    return answer.strip()


In [19]:
question = "how many laptop are available in you shop?"

answer = rag_chat(question)

print(answer)

We currently have a total of 27 laptop units available in stock across the following models:

* Lenovo Legion Pro 5: 15 available
* Apple MacBook Air M3: 8 available
* HP Spectre x360 14: 4 available
* Dell XPS 15 9530: 0 available (Currently out of stock)
